# 06 · Modelvergelijking en foutanalyse

**Auteurs:** _vul namen en concrete bijdragen in vóór indiening_  
**Doel:** De winnaar toetsen op tijd, subgroepen en concrete grote fouten voordat deployment volgt.

Elke codecel wordt voorafgegaan door uitleg. Voer de notebooks in nummervolgorde uit
vanaf de repository-root. Resultaten moeten door het team zelf worden gecontroleerd en
geïnterpreteerd; synthetische CI-data gelden nooit als onderzoeksbewijs.

In [ ]:
# Maak lokale projectimports betrouwbaar in VS Code, JupyterLab en nbconvert.
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").exists() and (path / "src").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Projectroot niet gevonden; open de map Brent als VS Code-workspace.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Projectroot: {PROJECT_ROOT}")

## Scorekaart

Een model moet de mediaanbaseline merkbaar verslaan en acceptabele staartfouten hebben.

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import MODEL_DIR, PROCESSED_DIR, REPORT_DIR, TARGET, WEATHER_PATH
from src.data.weather import attach_hourly_weather
from src.features.build import make_features
from src.models.train import temporal_split

metrics = pd.read_csv(REPORT_DIR / "metrics.csv").sort_values("mae_minutes")
display(metrics)
ax = metrics.plot.barh(x="model", y="mae_minutes", legend=False, color="#4f8f67")
ax.set(xlabel="hold-out MAE (minuten)", ylabel="", title="Lagere fout is beter")
plt.show()

## Residuen van het geselecteerde model

We reconstrueren exact dezelfde chronologische hold-outset en bewaren de grootste fouten voor bespreking.

In [ ]:
bundle = joblib.load(MODEL_DIR / "duration_model.joblib")
columns = [
    "started_at", "rideable_type", "member_casual", "start_station_id",
    "end_station_id", "start_lat", "start_lng", "end_lat", "end_lng", TARGET,
]
frame = pd.read_parquet(PROCESSED_DIR / "trips.parquet", columns=columns).sort_values("started_at")
if len(frame) > 500_000:
    frame = frame.iloc[np.linspace(0, len(frame)-1, 500_000, dtype=int)]
frame = attach_hourly_weather(frame, WEATHER_PATH)
_, test = temporal_split(frame)
test = test.copy()
test["prediction"] = np.maximum(0, bundle["model"].predict(make_features(test)))
test["error"] = test.prediction - test[TARGET]
test["absolute_error"] = test.error.abs()
display(test.nlargest(20, "absolute_error"))

## Subgroepanalyse

Gemiddelden kunnen een zwakke groep verbergen. We vergelijken rijderstype, fiets en weekend versus werkdag.

In [ ]:
test["weekpart"] = np.where(test.started_at.dt.dayofweek >= 5, "weekend", "weekday")
subgroup = test.groupby(["member_casual", "rideable_type", "weekpart"], observed=True).agg(
    n=("absolute_error", "size"),
    mae=("absolute_error", "mean"),
    median_ae=("absolute_error", "median"),
    bias=("error", "mean"),
).reset_index().query("n >= 100")
display(subgroup.sort_values("mae", ascending=False))

## Stabiliteit door de tijd

Maandelijkse MAE signaleert seizoensdrift of een problematische laatste periode.

In [ ]:
test["month"] = test.started_at.dt.to_period("M").astype(str)
monthly_error = test.groupby("month").agg(n=("absolute_error","size"), mae=("absolute_error","mean"), bias=("error","mean")).reset_index()
display(monthly_error)
sns.lineplot(data=monthly_error, x="month", y="mae", marker="o")
plt.xticks(rotation=45); plt.title("Hold-out MAE door de tijd"); plt.tight_layout()

## Definitief besluit

Accepteer de automatisch gekozen winnaar alleen als die de baseline verslaat, de winst
niet door één kleine groep komt, bias beperkt is en de API-latency praktisch blijft.
Beschrijf hier de concrete score, baselineverbetering, zwakste subgroup en vermoedelijke
oorzaak (ontbrekend weer/evenement/route). Dit is de model card voor de presentatie.